# Emirates NBD IFRS S1/S2 Style Extraction — Azure OpenAI Debug Notebook

This version is tuned for the actual uploaded Emirates NBD PDF.

Key parsing decisions:
- Uses **PyMuPDF (`fitz`) text extraction as primary**, because it preserves spaces much better for this PDF.
- Uses fixed report page ranges from the Contents page, skipping:
  - cover pages,
  - the Table of Contents,
  - section divider pages,
  - appendices.
- Builds style chunks only from the actual narrative pages:
  - General Requirements: pages 5-7
  - Governance: pages 9-18
  - Strategy: pages 20-46
  - Risk management: pages 48-54
  - Metrics and targets: pages 56-63

Output: `emirates_nbd_style_reference.json`

In [ ]:
# Optional installs
# Run this only if the packages are missing in your environment.
# !pip install pymupdf pdfplumber openai python-dotenv tqdm

In [ ]:
# ── Imports and logging ─────────────────────────────────────
import os
import re
import json
import logging
from pathlib import Path
from typing import Optional

import pdfplumber
from dotenv import load_dotenv
from tqdm import tqdm

try:
    import fitz  # PyMuPDF
except ImportError:
    fitz = None

try:
    from openai import AzureOpenAI
except ImportError:
    AzureOpenAI = None

load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
log = logging.getLogger(__name__)

print("Imports ready")
print("PyMuPDF available:", fitz is not None)
print("Azure OpenAI package available:", AzureOpenAI is not None)

In [3]:
# ── Configuration ───────────────────────────────────────────
# Update these paths for your local machine if needed.

PDF_PATH_CANDIDATES = [
    Path.cwd() / "emirates_nbd_group_2024_ifrs_s1_s2.pdf",
    Path.cwd().parent / "emirates_nbd_group_2024_ifrs_s1_s2.pdf",
    Path.cwd() / "data" / "emirates_nbd_group_2024_ifrs_s1_s2.pdf",
    Path.cwd().parent / "data" / "emirates_nbd_group_2024_ifrs_s1_s2.pdf",
    Path("/mnt/data") / "emirates_nbd_group_2024_ifrs_s1_s2.pdf",
]

PDF_PATH = next((p for p in PDF_PATH_CANDIDATES if p.exists()), None)

# Manual override example:
# PDF_PATH = Path(r"C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\emirates_nbd_group_2024_ifrs_s1_s2.pdf")

OUTPUT_PATH = Path("emirates_nbd_style_reference.json")
DEBUG = True

# ── Azure OpenAI settings ───────────────────────────────────
# Recommended .env variables:
# AZURE_OPENAI_API_KEY=...
# AZURE_OPENAI_ENDPOINT=https://<your-resource>.openai.azure.com/
# AZURE_OPENAI_API_VERSION=2024-10-21
# AZURE_OPENAI_DEPLOYMENT=<your-chat-deployment-name>
#
# Optional alternative env names are also supported:
# AZURE_OPENAI_CHAT_DEPLOYMENT, AZURE_OPENAI_MODEL, AZURE_OPENAI_DEPLOYMENT_NAME

AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_JUDGE_URL")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-21")
AZURE_OPENAI_DEPLOYMENT = (
    os.getenv("AZURE_OPENAI_DEPLOYMENT")
    or os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
    or os.getenv("AZURE_OPENAI_MODEL")
    or os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")
)

# Manual override examples:
# AZURE_OPENAI_ENDPOINT = "https://your-resource.openai.azure.com/"
# AZURE_OPENAI_API_KEY = "paste-your-key-here"   # Not recommended to hard-code in shared notebooks
# AZURE_OPENAI_DEPLOYMENT = "gpt-4.1"            # Your Azure deployment name, not necessarily the base model name
# AZURE_OPENAI_API_VERSION = "2024-10-21"

print("PDF_PATH:", PDF_PATH)
print("OUTPUT_PATH:", OUTPUT_PATH.resolve())
print("DEBUG:", DEBUG)
print("Azure endpoint configured:", bool(AZURE_OPENAI_ENDPOINT))
print("Azure key configured:", bool(AZURE_OPENAI_API_KEY))
print("Azure deployment:", AZURE_OPENAI_DEPLOYMENT)
print("Azure API version:", AZURE_OPENAI_API_VERSION)

PDF_PATH: None
OUTPUT_PATH: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\emirates_nbd_style_reference.json
DEBUG: True
Azure endpoint configured: True
Azure key configured: True
Azure deployment: None
Azure API version: 2024-07-18


In [ ]:
# ── Section detection config ────────────────────────────────
# For this specific Emirates NBD PDF, fixed page ranges are more reliable than keyword grouping.
# Page numbers are PDF-visible page numbers / one-indexed PDF pages.

USE_FIXED_EMIRATES_PAGE_RANGES = True

EMIRATES_SECTION_PAGE_RANGES = {
    # Skip page 4 because it is a section divider image page.
    "general": (5, 7),

    # Skip page 8 because it is a section divider image page.
    "governance": (9, 18),

    # Skip page 19 because it is a section divider image page.
    "strategy": (20, 46),

    # Skip page 47 because it is a section divider image page.
    "risk_management": (48, 54),

    # Skip page 55 because it is a section divider image page.
    "metrics_and_targets": (56, 63),
}

SECTION_ORDER = [
    "general",
    "governance",
    "strategy",
    "risk_management",
    "metrics_and_targets",
]

SECTION_DISPLAY_NAMES = {
    "general": "General Requirements",
    "governance": "Governance",
    "strategy": "Strategy",
    "risk_management": "Risk management",
    "metrics_and_targets": "Metrics and targets",
}

# Used only for fallback/debug.
SECTION_KEYWORDS = {
    "governance": [
        "governance", "board oversight", "board of directors",
        "management role", "committee", "oversight"
    ],
    "strategy": [
        "strategy", "strategic", "climate-related risks", "opportunities",
        "scenario analysis", "transition plan", "resilience",
        "time horizon", "physical risk", "transition risk"
    ],
    "risk_management": [
        "risk management", "risk identification", "risk assessment",
        "enterprise risk", "climate risk integration", "risk appetite",
        "risk framework"
    ],
    "metrics_and_targets": [
        "metrics", "targets", "scope 1", "scope 2", "scope 3",
        "ghg emissions", "carbon", "tco2e", "net zero",
        "financed emissions", "baseline year", "emissions intensity"
    ]
}

SKIP_PAGE_PATTERNS = [
    r"^table of contents",
    r"^contents$",
    r"^appendix",
    r"^\d+$",
]

MAX_CHUNK_WORDS = 1800

print("Fixed Emirates section ranges ready")
print(json.dumps(EMIRATES_SECTION_PAGE_RANGES, indent=2))

In [ ]:
# ── Step 1: PDF page extraction and cleaning ────────────────
def clean_pdf_text(text: str) -> str:
    """
    Light text cleaner. PyMuPDF already preserves spaces well for this PDF.
    """
    if not text:
        return ""

    text = text.replace("\x08", " ")
    text = text.replace("\u00a0", " ")
    text = text.replace("\u2002", " ")
    text = text.replace("\u2003", " ")
    text = text.replace("￾", "-")

    # Normalise line endings and repeated whitespace, but preserve paragraph lines.
    cleaned_lines = []
    for line in text.splitlines():
        line = re.sub(r"[ \t]+", " ", line).strip()
        if line:
            cleaned_lines.append(line)

    text = "\n".join(cleaned_lines)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def _extract_pages_with_pymupdf(pdf_path: Path) -> list[dict]:
    """
    Primary extractor for this Emirates NBD PDF.
    PyMuPDF's text layer is much cleaner than pdfplumber for this file.
    """
    if fitz is None:
        return []

    pages = []
    doc = fitz.open(str(pdf_path))

    for i, page in enumerate(tqdm(doc, desc="Extracting pages with PyMuPDF")):
        text = page.get_text("text", sort=True) or ""
        text = clean_pdf_text(text)

        if not text or len(text) < 50:
            continue

        pages.append({
            "page_num": i + 1,
            "text": text,
            "word_count": len(text.split()),
            "extractor": "pymupdf_text_sort_true",
        })

    doc.close()
    return pages


def _extract_pages_with_pdfplumber_words(pdf_path: Path) -> list[dict]:
    """
    Fallback extractor only. For this PDF, PyMuPDF is preferred.
    """
    pages = []

    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(tqdm(pdf.pages, desc="Fallback extracting pages with pdfplumber words")):
            words = page.extract_words(
                x_tolerance=2,
                y_tolerance=3,
                keep_blank_chars=False,
                use_text_flow=False,
            )

            # Simple visual reconstruction.
            words = sorted(words, key=lambda w: (round(float(w.get("top", 0)), 1), float(w.get("x0", 0))))
            lines = []
            current = []
            current_top = None

            for w in words:
                top = float(w.get("top", 0))
                txt = str(w.get("text", "")).replace("\x08", " ").strip()
                if not txt:
                    continue

                if current_top is None or abs(top - current_top) <= 3:
                    current.append(txt)
                    current_top = top if current_top is None else current_top
                else:
                    lines.append(" ".join(current))
                    current = [txt]
                    current_top = top

            if current:
                lines.append(" ".join(current))

            text = clean_pdf_text("\n".join(lines))

            if not text or len(text) < 50:
                continue

            pages.append({
                "page_num": i + 1,
                "text": text,
                "word_count": len(text.split()),
                "extractor": "pdfplumber_words_fallback",
            })

    return pages


def looks_garbled_or_concatenated(text: str) -> bool:
    """
    Detect badly extracted text.
    We expect some company names/URLs to be long, so this is only a warning.
    """
    if not text:
        return True

    # Examples of bad extraction: completeparagraphwithoutspaces or GGeenneerraall.
    very_long_alpha_tokens = re.findall(r"\b[A-Za-z]{28,}\b", text)
    repeated_heading_pairs = len(re.findall(r"([A-Za-z])\1{2,}", text))

    return len(very_long_alpha_tokens) >= 3 or repeated_heading_pairs > 30


def is_table_of_contents_page(text: str) -> bool:
    """
    Detect TOC/Contents pages.
    For this PDF, page 3 is the main contents page.
    """
    t = clean_pdf_text(text).lower()
    first_2500 = t[:2500]

    toc_signals = [
        "contents",
        "general requirements",
        "governance",
        "strategy",
        "risk management",
        "metrics and targets",
        "appendix",
    ]

    signal_count = sum(s in first_2500 for s in toc_signals)
    has_many_section_numbers = len(re.findall(r"\b[1-5]\.\d+\b", first_2500)) >= 8

    return ("contents" in first_2500 and signal_count >= 4) or has_many_section_numbers


def is_section_divider_page(page_num: int, text: str) -> bool:
    """
    Divider pages are visual title pages: 4, 8, 19, 47, 55, 64, 67.
    They are useful for structure but not for style extraction.
    """
    known_dividers = {1, 2, 4, 8, 19, 47, 55, 64, 67}
    if page_num in known_dividers:
        return True

    t = clean_pdf_text(text).lower()
    if len(t.split()) <= 8 and any(label.lower() in t for label in SECTION_DISPLAY_NAMES.values()):
        return True

    return False


def should_skip_page(page: dict, debug: bool = False) -> bool:
    """
    Skip cover, TOC, section divider, appendix cover, and very low-content pages.
    """
    page_num = page["page_num"]
    text = clean_pdf_text(page["text"])

    if not text:
        return True

    if is_table_of_contents_page(text):
        return True

    if is_section_divider_page(page_num, text):
        return True

    first_line = text.split("\n")[0].lower().strip()
    if any(re.match(p, first_line) for p in SKIP_PAGE_PATTERNS):
        return True

    return False


def extract_pdf_pages(pdf_path: str | Path, debug: bool = False) -> list[dict]:
    """
    Extract pages from the uploaded Emirates NBD PDF.
    """
    pdf_path = Path(pdf_path)
    log.info(f"Opening PDF: {pdf_path}")

    pages = _extract_pages_with_pymupdf(pdf_path)

    if not pages:
        log.warning("PyMuPDF extraction unavailable or empty. Falling back to pdfplumber.")
        pages = _extract_pages_with_pdfplumber_words(pdf_path)

    before_skip = len(pages)
    pages = [p for p in pages if not should_skip_page(p, debug=debug)]

    log.info(f"Extracted {len(pages)} content pages after skipping ({before_skip - len(pages)} skipped)")
    if pages:
        log.info(f"Extractor used: {pages[0].get('extractor')}")

    bad_pages = [p["page_num"] for p in pages if looks_garbled_or_concatenated(p["text"])]
    if bad_pages:
        print("WARNING: possible concatenated/garbled text on pages:", bad_pages[:20])

    return pages


print("PyMuPDF-first PDF extraction ready")

In [ ]:
# ── Run Step 1: Extract pages ───────────────────────────────
if PDF_PATH is None:
    raise FileNotFoundError(
        "PDF_PATH is None. Set PDF_PATH_MANUAL in the Configuration cell, "
        "or set EMIRATES_NBD_PDF_PATH in your .env file."
    )

pages = extract_pdf_pages(PDF_PATH, debug=DEBUG)

print("Pages extracted:", len(pages))
print("First 5 extracted pages:")
for page in pages[:5]:
    first = page["text"].splitlines()[0] if page["text"].splitlines() else ""
    print(f"- page {page['page_num']} | words={page['word_count']} | extractor={page.get('extractor')} | first line={first[:120]}")

print("\nSample text quality check:")
for page in pages[:3]:
    print("\n" + "="*70)
    print(f"PAGE {page['page_num']}")
    print("="*70)
    print(page["text"][:900])

In [ ]:
# ── Step 2: Fixed page-range section grouping ───────────────
def classify_page_section(page_text: str) -> Optional[str]:
    """
    Fallback classifier based on keyword density.
    Fixed ranges are used for the actual Emirates NBD PDF.
    """
    text_lower = clean_pdf_text(page_text).lower()
    scores = {}

    for section, keywords in SECTION_KEYWORDS.items():
        score = sum(text_lower.count(kw) for kw in keywords)
        if score > 0:
            scores[section] = score

    if not scores:
        return None

    return max(scores, key=scores.get)


def classify_page_scores(page_text: str) -> dict:
    """
    Debug helper: return all keyword scores.
    """
    text_lower = clean_pdf_text(page_text).lower()
    return {
        section: sum(text_lower.count(kw) for kw in keywords)
        for section, keywords in SECTION_KEYWORDS.items()
    }


def group_pages_by_fixed_ranges(pages: list[dict]) -> dict[str, list[dict]]:
    """
    Group pages using known Emirates NBD report ranges from the Contents page.
    """
    pages_by_num = {p["page_num"]: p for p in pages}
    sections = {s: [] for s in SECTION_ORDER}
    sections["unclassified"] = []

    assigned = set()

    for section, (start, end) in EMIRATES_SECTION_PAGE_RANGES.items():
        selected = []
        for page_num in range(start, end + 1):
            page = pages_by_num.get(page_num)
            if page is not None:
                selected.append(page)
                assigned.add(page_num)
        sections[section] = selected

    sections["unclassified"] = [
        p for p in pages
        if p["page_num"] not in assigned
    ]

    return sections


def group_pages_by_keyword_fallback(pages: list[dict]) -> dict[str, list[dict]]:
    """
    Fallback only if you use another PDF without the Emirates page ranges.
    """
    sections = {s: [] for s in SECTION_ORDER}
    sections["unclassified"] = []

    for page in pages:
        section = classify_page_section(page["text"])
        if section:
            sections[section].append(page)
        else:
            sections["unclassified"].append(page)

    return sections


def group_pages_by_section(pages: list[dict]) -> dict[str, list[dict]]:
    """
    Main grouping function used by the notebook.
    """
    if USE_FIXED_EMIRATES_PAGE_RANGES:
        sections = group_pages_by_fixed_ranges(pages)
        print("Using fixed Emirates page ranges from the Contents page.")
    else:
        sections = group_pages_by_keyword_fallback(pages)
        print("Using keyword fallback grouping.")

    print("\nSection page ranges/counts:")
    for name, pages_list in sections.items():
        if pages_list:
            page_nums = [p["page_num"] for p in pages_list]
            print(f"- {name}: {len(pages_list)} pages | {min(page_nums)}-{max(page_nums)} | pages={page_nums[:12]}{'...' if len(page_nums) > 12 else ''}")
        else:
            print(f"- {name}: 0 pages")

    return sections


print("Fixed page-range section grouping ready")

In [ ]:
# ── Run Step 2: Group pages by section ──────────────────────
section_pages = group_pages_by_section(pages)

print("\nSection page counts:")
for section, pg_list in section_pages.items():
    print(f"- {section}: {len(pg_list)} pages")

print("\nPage quality debug sample:")
for page in pages[:10]:
    first = page["text"].splitlines()[0][:90] if page["text"].splitlines() else ""
    scores = classify_page_scores(page["text"])
    print(f"page {page['page_num']:>3} | scores={scores} | first='{first}'")

In [ ]:
# ── Step 3: Build section chunks ────────────────────────────
def remove_likely_headers_footers(text: str) -> str:
    """
    Remove recurring low-value page headers/footers from chunks.
    """
    cleaned = []

    for line in text.splitlines():
        l = line.strip()
        ll = l.lower()

        if not l:
            continue
        if re.fullmatch(r"\d+", l):
            continue
        if ll.startswith("emirates nbd group 2024 ifrs s1 and s2 report"):
            continue
        if ll in {"governance", "strategy", "risk management", "metrics and targets", "general requirements"}:
            # Keep the first occurrence later if useful, but avoid repeated page headers.
            continue

        cleaned.append(l)

    return "\n".join(cleaned)


def build_section_chunk(pages: list[dict], section: str | None = None, max_words: int = MAX_CHUNK_WORDS) -> str:
    """
    Concatenate cleaned section pages up to max_words.
    """
    if not pages:
        return ""

    combined_text = "\n\n".join(clean_pdf_text(page["text"]) for page in pages)
    combined_text = remove_likely_headers_footers(combined_text)

    # Add a clear section label for the style model.
    if section:
        heading = SECTION_DISPLAY_NAMES.get(section, section)
        combined_text = f"{heading}\n\n{combined_text}"

    words = combined_text.split()
    if len(words) > max_words:
        combined_text = " ".join(words[:max_words])

    return combined_text.strip()


print("build_section_chunk() ready")

In [ ]:
# ── Run Step 3: Build chunks ────────────────────────────────
section_chunks = {}

# Analyze only the real report sections.
sections_to_analyze = [
    "general",
    "governance",
    "strategy",
    "risk_management",
    "metrics_and_targets",
]

for section in sections_to_analyze:
    pg_list = section_pages.get(section, [])
    if pg_list:
        chunk = build_section_chunk(pg_list, section=section, max_words=MAX_CHUNK_WORDS)
        section_chunks[section] = chunk
        page_nums = [p["page_num"] for p in pg_list]
        print(f"{section}: {len(chunk.split())} words | pages {page_nums}")
    else:
        print(f"{section}: 0 pages -> no chunk")

print("\nChunk preview:")
for section, chunk in section_chunks.items():
    print("\n" + "="*80)
    print(section.upper())
    print("="*80)
    print(chunk[:1400])

## Diagnostic: chunk quality check


In [ ]:
# ── Diagnostic: check chunk quality before calling Azure ─────────
def chunk_quality_report(section_chunks: dict) -> dict:
    report = {}

    for section, chunk in section_chunks.items():
        long_unspaced = re.findall(r"\b[A-Za-z]{28,}\b", chunk)
        repeated_artifacts = re.findall(r"([A-Za-z])\1{2,}", chunk)

        report[section] = {
            "word_count": len(chunk.split()),
            "starts_with": chunk[:160],
            "long_unspaced_words_count": len(long_unspaced),
            "long_unspaced_words_sample": long_unspaced[:10],
            "repeated_letter_artifact_count": len(repeated_artifacts),
            "looks_like_table_of_contents": is_table_of_contents_page(chunk),
            "looks_garbled_or_concatenated": looks_garbled_or_concatenated(chunk),
        }

    return report

quality = chunk_quality_report(section_chunks)
print(json.dumps(quality, indent=2, ensure_ascii=False))

In [ ]:
# ── Step 4: Azure OpenAI section analysis prompts ─────────────────
SECTION_PROMPTS = {
    "governance": """
You are a sustainability reporting analyst. Analyze this GOVERNANCE section from an IFRS S1/S2 bank sustainability report.

Extract the following and return ONLY valid JSON — no preamble, no markdown fences:

{
  "section": "governance",
  "tone_descriptors": ["list of 4-6 adjectives describing the writing tone"],
  "sentence_structure": "short paragraph describing typical sentence length and complexity",
  "example_sentences": ["3 verbatim sentences that best represent the reporting style"],
  "opening_patterns": ["2-3 ways this section typically opens a paragraph"],
  "board_language": ["exact phrases used to describe board/committee oversight roles"],
  "formality_markers": ["list of formal phrases/constructions used, e.g. 'The Board has established...'"],
  "tense_usage": "present/past/mixed — and how each is used",
  "passive_vs_active": "ratio estimate and pattern description",
  "hedging_phrases": ["phrases used to soften or qualify statements"],
  "section_length_estimate": "estimated word count range for this section"
}

REPORT TEXT:
{text}
""",

    "strategy": """
You are a sustainability reporting analyst. Analyze this STRATEGY section from an IFRS S1/S2 bank sustainability report.

Extract the following and return ONLY valid JSON — no preamble, no markdown fences:

{
  "section": "strategy",
  "tone_descriptors": ["list of 4-6 adjectives describing the writing tone"],
  "sentence_structure": "short paragraph describing typical sentence length and complexity",
  "example_sentences": ["3 verbatim sentences that best represent the reporting style"],
  "risk_language": {
    "physical_risk_phrases": ["exact phrases used to describe physical climate risks"],
    "transition_risk_phrases": ["exact phrases used to describe transition risks"],
    "opportunity_phrases": ["phrases used to describe climate opportunities"]
  },
  "time_horizon_language": ["how short/medium/long-term horizons are expressed in text"],
  "scenario_analysis_phrases": ["exact phrases used when referencing scenario analysis"],
  "forward_looking_language": ["hedged phrases for forward-looking statements, e.g. 'The Group expects...'"],
  "financial_impact_language": ["how financial impacts are described without specific numbers"],
  "formality_markers": ["key formal phrases or constructions"],
  "section_length_estimate": "estimated word count range for this section"
}

REPORT TEXT:
{text}
""",

    "risk_management": """
You are a sustainability reporting analyst. Analyze this RISK MANAGEMENT section from an IFRS S1/S2 bank sustainability report.

Extract the following and return ONLY valid JSON — no preamble, no markdown fences:

{
  "section": "risk_management",
  "tone_descriptors": ["list of 4-6 adjectives describing the writing tone"],
  "sentence_structure": "short paragraph describing typical sentence length and complexity",
  "example_sentences": ["3 verbatim sentences that best represent the reporting style"],
  "process_description_language": ["phrases used to describe identification/assessment processes"],
  "integration_language": ["how climate risk integration into ERM is expressed"],
  "risk_classification_language": ["how risk categories or tiers are described"],
  "governance_link_phrases": ["phrases that connect risk management back to governance"],
  "formality_markers": ["key formal phrases or constructions"],
  "hedging_phrases": ["phrases used to qualify risk statements"],
  "section_length_estimate": "estimated word count range for this section"
}

REPORT TEXT:
{text}
""",

    "metrics_and_targets": """
You are a sustainability reporting analyst. Analyze this METRICS AND TARGETS section from an IFRS S1/S2 bank sustainability report.

Extract the following and return ONLY valid JSON — no preamble, no markdown fences:

{
  "section": "metrics_and_targets",
  "tone_descriptors": ["list of 4-6 adjectives describing the writing tone"],
  "sentence_structure": "short paragraph describing typical sentence length and complexity",
  "example_sentences": ["3 verbatim sentences that best represent the reporting style"],
  "emissions_reporting_style": {
    "scope_1_phrasing": "how scope 1 is introduced and described",
    "scope_2_phrasing": "how scope 2 is introduced and described",
    "scope_3_phrasing": "how scope 3 is introduced and described",
    "financed_emissions_phrasing": "how financed emissions are described if present"
  },
  "target_description_pattern": "how targets are structured in text (baseline → progress → goal)",
  "unit_presentation": ["how units like tCO2e, MWh are presented inline"],
  "methodology_citation_style": "how calculation methodologies are referenced",
  "table_introduction_phrases": ["phrases used to introduce data tables"],
  "year_on_year_comparison_language": ["phrases used for YoY comparisons"],
  "formality_markers": ["key formal phrases or constructions"],
  "section_length_estimate": "estimated word count range for this section"
}

REPORT TEXT:
{text}
""",

    "general": """
You are a sustainability reporting analyst. Analyze this text from an IFRS S1/S2 bank sustainability report.

Extract general style patterns and return ONLY valid JSON — no preamble, no markdown fences:

{
  "section": "general",
  "report_voice": "first-person plural / third-person / mixed — with description",
  "organization_reference_style": "how the bank refers to itself (e.g. 'the Group', 'Emirates NBD', 'we')",
  "paragraph_length": "typical paragraph word count range",
  "list_usage": "how bullet points and numbered lists are used — sparingly/frequently/never",
  "cross_reference_style": "how other sections or documents are cross-referenced",
  "boilerplate_phrases": ["5-8 recurring formal phrases found throughout the report"],
  "disclosure_disclaimer_language": ["phrases used for regulatory disclaimer statements"],
  "year_reference_style": "how reporting year is referred to (e.g. '2024', 'the reporting period', 'the year under review')",
  "document_title_style": "how section and subsection titles are formatted"
}

REPORT TEXT:
{text}
"""
}

print("SECTION_PROMPTS ready:", list(SECTION_PROMPTS.keys()))

In [ ]:
# ── Step 5: Azure OpenAI API helper ─────────────────────────
def parse_json_response(raw: str) -> dict:
    '''
    Strip accidental markdown fences and parse JSON.
    '''
    raw = raw.strip()
    raw = re.sub(r"^```json\s*", "", raw)
    raw = re.sub(r"^```\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw)
    return json.loads(raw)


def get_azure_openai_client() -> AzureOpenAI:
    '''
    Build Azure OpenAI client from environment variables or manual config cell values.
    '''
    if AzureOpenAI is None:
        raise ImportError("openai package is not installed. Run the install cell first.")

    missing = []
    if not AZURE_OPENAI_API_KEY:
        missing.append("AZURE_OPENAI_API_KEY")
    if not AZURE_OPENAI_ENDPOINT:
        missing.append("AZURE_OPENAI_ENDPOINT")
    if not AZURE_OPENAI_DEPLOYMENT:
        missing.append("AZURE_OPENAI_DEPLOYMENT or AZURE_OPENAI_CHAT_DEPLOYMENT")

    if missing:
        raise EnvironmentError(
            "Missing Azure OpenAI settings: " + ", ".join(missing) +
            ". Set them in your .env file or in the configuration cell."
        )

    return AzureOpenAI(
        api_key=AZURE_OPENAI_API_KEY,
        azure_endpoint=AZURE_OPENAI_ENDPOINT,
        api_version=AZURE_OPENAI_API_VERSION,
    )


def azure_chat_json(
    client: AzureOpenAI,
    system_prompt: str,
    user_prompt: str,
    max_tokens: int = 2000,
    temperature: float = 0,
) -> dict:
    '''
    Calls Azure OpenAI and returns parsed JSON.
    Uses JSON response format when supported; falls back to normal text JSON if needed.
    '''
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    try:
        response = client.chat.completions.create(
            model=AZURE_OPENAI_DEPLOYMENT,
            messages=messages,
            max_tokens=max_tokens,
            temperature=temperature,
            response_format={"type": "json_object"},
        )
    except Exception as first_error:
        # Some Azure deployments/API versions may not support response_format.
        log.warning(f"Retrying without response_format because first call failed: {first_error}")
        response = client.chat.completions.create(
            model=AZURE_OPENAI_DEPLOYMENT,
            messages=messages,
            max_tokens=max_tokens,
            temperature=temperature,
        )

    raw = response.choices[0].message.content.strip()
    return parse_json_response(raw)


def analyze_section_with_azure(
    client: AzureOpenAI,
    section_name: str,
    text_chunk: str,
    debug: bool = False
) -> Optional[dict]:
    '''
    Send a section chunk to Azure OpenAI for style analysis.
    Returns parsed JSON dict or None on failure.
    '''
    if not text_chunk.strip():
        log.warning(f"Empty chunk for section '{section_name}' — skipping")
        return None

    prompt_template = SECTION_PROMPTS.get(section_name, SECTION_PROMPTS["general"])
    prompt = prompt_template.format(text=text_chunk)

    if debug:
        log.info(f"Sending {len(text_chunk.split())} words to Azure OpenAI for '{section_name}'")

    try:
        parsed = azure_chat_json(
            client=client,
            system_prompt=(
                "You are a document style analyst. "
                "You always respond with valid JSON only — no markdown, no preamble, no explanation. "
                "If you cannot extract a field, use null. Never invent data not present in the text."
            ),
            user_prompt=prompt,
            max_tokens=2000,
            temperature=0,
        )

        log.info(f"  ✓ '{section_name}' analysis complete")
        return parsed

    except json.JSONDecodeError as e:
        log.error(f"JSON parse error for '{section_name}': {e}")
        return None
    except Exception as e:
        log.error(f"Azure OpenAI API error for '{section_name}': {e}")
        return None


print("Azure OpenAI helper functions ready")

In [ ]:
# ── Run Step 5: Analyze one section first for debugging ─────
# Start with one section before running all sections.
# Change TEST_SECTION if needed.

TEST_SECTION = "governance"

client = get_azure_openai_client()

test_analysis = analyze_section_with_azure(
    client=client,
    section_name=TEST_SECTION,
    text_chunk=section_chunks.get(TEST_SECTION, ""),
    debug=DEBUG,
)

print(json.dumps(test_analysis, indent=2, ensure_ascii=False) if test_analysis else "No analysis returned")

In [ ]:
# ── Run Step 5b: Analyze all sections ───────────────────────
# Run after the test section works.

section_analyses = {}

for section, chunk in section_chunks.items():
    print("\n" + "="*80)
    print("Analyzing:", section)
    print("="*80)

    analysis = analyze_section_with_azure(
        client=client,
        section_name=section,
        text_chunk=chunk,
        debug=DEBUG,
    )
    section_analyses[section] = analysis

print("\nCompleted analyses:")
for section, analysis in section_analyses.items():
    print(f"- {section}: {'OK' if analysis else 'FAILED'}")

In [ ]:
# ── Step 6: Synthesis prompt ────────────────────────────────
SYNTHESIS_PROMPT = """
You are a document style consultant. You have analyzed sections of an IFRS S1/S2 sustainability report.

Here are the per-section style analyses:
{section_analyses}

Synthesize these into a unified style guide for use by AI agents generating new report sections.

Return ONLY valid JSON — no preamble, no markdown fences:

{
  "report_identity": {
    "organization_name": "how the bank refers to itself in text",
    "reporting_year": "year or period reference style used",
    "document_type": "type of document style (formal corporate report, etc.)"
  },
  "universal_style_rules": [
    "8-10 rules that apply across ALL sections, written as actionable instructions for an AI writer"
  ],
  "tone_profile": {
    "primary_tone": "single best descriptor",
    "secondary_tones": ["2-3 secondary descriptors"],
    "what_to_avoid": ["3-5 tone/style anti-patterns NOT present in this report"]
  },
  "section_specific_instructions": {
    "governance": "2-3 sentence writing instruction for governance section agent",
    "strategy": "2-3 sentence writing instruction for strategy section agent",
    "risk_management": "2-3 sentence writing instruction for risk management section agent",
    "metrics_and_targets": "2-3 sentence writing instruction for metrics section agent"
  },
  "common_opening_patterns": ["5 ways sections or paragraphs typically begin"],
  "common_closing_patterns": ["3 ways sections or paragraphs typically end"],
  "master_phrase_bank": ["20-25 exact phrases or sentence starters from the report to reuse in generation"],
  "formatting_conventions": {
    "heading_style": "description of heading formatting",
    "table_style": "description of how tables are presented",
    "list_style": "description of bullet/numbered list usage",
    "number_formatting": "how numbers and percentages are written"
  }
}
"""

print("SYNTHESIS_PROMPT ready")

In [ ]:
# ── Step 6: Synthesize unified style guide ──────────────────
def synthesize_style_guide(client: AzureOpenAI, section_analyses: dict) -> Optional[dict]:
    '''
    Call Azure OpenAI to synthesize all section analyses into a unified style guide.
    '''
    log.info("Synthesizing unified style guide...")

    valid_analyses = {k: v for k, v in section_analyses.items() if v is not None}

    if not valid_analyses:
        log.error("No valid section analyses to synthesize")
        return None

    prompt = SYNTHESIS_PROMPT.format(
        section_analyses=json.dumps(valid_analyses, indent=2, ensure_ascii=False)
    )

    try:
        parsed = azure_chat_json(
            client=client,
            system_prompt=(
                "You are a document style consultant. "
                "Respond with valid JSON only. No markdown, no preamble."
            ),
            user_prompt=prompt,
            max_tokens=3000,
            temperature=0,
        )

        log.info("  ✓ Synthesis complete")
        return parsed

    except json.JSONDecodeError as e:
        log.error(f"Synthesis JSON parse error: {e}")
        return None
    except Exception as e:
        log.error(f"Synthesis Azure OpenAI API error: {e}")
        return None


style_guide = synthesize_style_guide(client, section_analyses)

print(json.dumps(style_guide, indent=2, ensure_ascii=False) if style_guide else "No style guide generated")

In [ ]:
# ── Step 7: Build LangGraph agent prompts ───────────────────
def build_agent_prompts(style_guide: dict) -> dict[str, str]:
    '''
    Build system prompts for each LangGraph section agent using extracted style patterns.
    '''
    if not style_guide:
        return {}

    universal_rules = "\n".join(
        f"- {rule}"
        for rule in style_guide.get("universal_style_rules", [])
    )

    phrase_bank = "\n".join(
        f'- "{phrase}"'
        for phrase in style_guide.get("master_phrase_bank", [])
    )

    org_name = style_guide.get("report_identity", {}).get("organization_name", "the Group")
    primary_tone = style_guide.get("tone_profile", {}).get("primary_tone", "formal")

    prompts = {}

    for section in ["governance", "strategy", "risk_management", "metrics_and_targets"]:
        section_instruction = (
            style_guide
            .get("section_specific_instructions", {})
            .get(section, "Write in a formal, professional tone.")
        )

        avoid = "\n".join(
            f"- {item}"
            for item in style_guide.get("tone_profile", {}).get("what_to_avoid", [])
        )

        prompts[section] = f'''You are an IFRS S1/S2 sustainability reporting specialist writing for {org_name}.

SECTION: {section.upper().replace("_", " ")}

STYLE INSTRUCTION:
{section_instruction}

UNIVERSAL STYLE RULES:
{universal_rules}

TONE: {primary_tone}

AVOID:
{avoid}

PHRASE BANK:
{phrase_bank}

TASK:
Given the JSON input data for the {section.replace("_", " ")} section, write the complete disclosure narrative.

Rules:
- Do not invent data not present in the input JSON.
- If a required disclosure is missing, state the evidence boundary in report-style wording.
- Match the style patterns above.
- Output the section text only.
'''

    return prompts


agent_prompts = build_agent_prompts(style_guide)

print("Agent prompts generated:", len(agent_prompts))
for section, prompt in agent_prompts.items():
    print("\n" + "="*80)
    print(section.upper())
    print("="*80)
    print(prompt[:1200])

In [ ]:
# ── Step 8: Assemble and save output ────────────────────────
output = {
    "metadata": {
        "source_pdf": str(Path(PDF_PATH).name) if PDF_PATH else None,
        "pages_extracted": len(pages),
        "sections_found": {k: len(v) for k, v in section_pages.items()},
        "extraction_model": "AZURE_OPENAI_DEPLOYMENT",
    },
    "per_section_analysis": section_analyses,
    "unified_style_guide": style_guide,
    "agent_prompts": agent_prompts,
}

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print("Saved output to:", OUTPUT_PATH.resolve())
print("Sections analyzed:", sum(1 for v in section_analyses.values() if v), "/", len(section_analyses))
print("Agent prompts:", list(agent_prompts.keys()))

In [ ]:
# ── Optional: Save each agent prompt as a separate .txt file ─
prompt_dir = Path("emirates_style_agent_prompts")
prompt_dir.mkdir(exist_ok=True)

for section, prompt in agent_prompts.items():
    path = prompt_dir / f"{section}_style_prompt.txt"
    path.write_text(prompt, encoding="utf-8")
    print("Saved:", path)

In [ ]:
# ── Optional: Reload and inspect final JSON ─────────────────
with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    ref = json.load(f)

print("Top-level keys:", list(ref.keys()))
print("Metadata:")
print(json.dumps(ref.get("metadata", {}), indent=2, ensure_ascii=False))

print("\nUniversal style rules:")
for i, rule in enumerate(ref.get("unified_style_guide", {}).get("universal_style_rules", []), 1):
    print(f"{i}. {rule}")